In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('telecom_customer_churn.csv')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 35 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customer_id           15000 non-null  str    
 1   age                   14550 non-null  float64
 2   gender                15000 non-null  str    
 3   marital_status        15000 non-null  str    
 4   education_level       15000 non-null  str    
 5   location_type         15000 non-null  str    
 6   tenure_months         15000 non-null  int64  
 7   contract_type         15000 non-null  str    
 8   payment_method        15000 non-null  str    
 9   paperless_billing     15000 non-null  str    
 10  auto_payment          15000 non-null  str    
 11  credit_score          14550 non-null  float64
 12  account_status        15000 non-null  str    
 13  total_day_minutes     15000 non-null  float64
 14  total_day_calls       15000 non-null  int64  
 15  total_eve_minutes     15000 no

In [4]:
# 'Unknown' for NaN + flag for missingness.

df['complaint_status'] = df['complaint_status'].fillna('Unknown')
df['complaint_status_flag'] = df['complaint_status'].isnull().astype(int)

In [5]:
# Handle missing / outlier age values 

df['age'] = pd.to_numeric(df['age'], errors='coerce')

age_median = df['age'].median()
df['age'] = df['age'].fillna(age_median)
df['age'] = df['age'].clip(lower=18, upper=90)

age_bins = [0, 25, 35, 50, 65, 120]
age_labels = ['<25', '25-34', '35-49', '50-64', '65+']


print('Age missing (after fill):', df['age'].isnull().sum())
print('Age median (used for fill):', age_median)
print('Age counts after fill:')
print(df['age'].describe())

Age missing (after fill): 0
Age median (used for fill): 45.0
Age counts after fill:
count    15000.000000
mean        44.737667
std         14.133333
min         18.000000
25%         35.000000
50%         45.000000
75%         54.000000
max         80.000000
Name: age, dtype: float64


In [6]:
# Credit score handling

df['credit_score'] = pd.to_numeric(df['credit_score'], errors='coerce')
credit_median = df['credit_score'].median()
df['credit_score'] = df['credit_score'].fillna(credit_median)
# realistic range check
df['credit_score'] = df['credit_score'].clip(lower=300, upper=850)

print(df['credit_score'].describe())

count    15000.000000
mean       679.272467
std         77.729558
min        360.000000
25%        628.000000
50%        679.000000
75%        732.000000
max        850.000000
Name: credit_score, dtype: float64


In [7]:
# satisfaction_score handling

# if text values like Low/Medium/High, map to numeric; otherwise numeric transform
if df['satisfaction_score'].dtype == 'object':
    mapping = {'low':1, 'medium':2, 'high':3, 'Very Low':1, 'Very High':4}
    df['satisfaction_score'] = df['satisfaction_score'].astype(str).str.strip().str.lower().map(mapping).astype('float')

# convert numeric and impute
df['satisfaction_score'] = pd.to_numeric(df['satisfaction_score'], errors='coerce')
sat_median = df['satisfaction_score'].median()
df['satisfaction_score'] = df['satisfaction_score'].fillna(sat_median)


print(df['satisfaction_score'].describe())

count    15000.000000
mean         6.795533
std          1.375059
min          1.000000
25%          6.000000
50%          7.000000
75%          8.000000
max         10.000000
Name: satisfaction_score, dtype: float64


In [8]:
# monthly_charges handling

# numeric conversion
df['monthly_charges'] = pd.to_numeric(df['monthly_charges'], errors='coerce')

if 'contract_type' in df.columns:
    monthly_medians = df.groupby('contract_type')['monthly_charges'].median()
    df['monthly_charges'] = df.apply(lambda r: monthly_medians.get(r['contract_type'], r['monthly_charges']) if pd.isna(r['monthly_charges']) else r['monthly_charges'], axis=1)

# fallback global median
monthly_median = df['monthly_charges'].median()
df['monthly_charges'] = df['monthly_charges'].fillna(monthly_median)

# ensure positive
df['monthly_charges'] = df['monthly_charges'].clip(lower=0)

print(df['monthly_charges'].describe())

count    15000.00000
mean        85.14824
std         34.83872
min         20.00000
25%         62.00750
50%         79.45000
75%        101.18250
max        450.00000
Name: monthly_charges, dtype: float64


In [9]:
print('Null counts for all columns:')
print(df.isnull().sum())


Null counts for all columns:
customer_id              0
age                      0
gender                   0
marital_status           0
education_level          0
location_type            0
tenure_months            0
contract_type            0
payment_method           0
paperless_billing        0
auto_payment             0
credit_score             0
account_status           0
total_day_minutes        0
total_day_calls          0
total_eve_minutes        0
total_eve_calls          0
total_night_minutes      0
total_night_calls        0
data_usage_gb            0
streaming_tv             0
streaming_movies         0
online_security          0
online_backup            0
device_protection        0
num_support_tickets      0
num_technical_issues     0
satisfaction_score       0
complaint_status         0
monthly_charges          0
total_charges            0
avg_call_charge          0
late_payment_count       0
signup_date              0
churn                    0
complaint_status_flag    0

In [10]:
display(df)

,customer_id,age,gender,marital_status,education_level,location_type,tenure_months,contract_type,payment_method,paperless_billing,...,num_technical_issues,satisfaction_score,complaint_status,monthly_charges,total_charges,avg_call_charge,late_payment_count,signup_date,churn,complaint_status_flag
0,CUST_00000001,52.0,Male,Married,High School,Suburban,34,Two year,Bank transfer,No,...,0,10.0,Resolved,53.42,1721.86,0.9893,0,2021-02-23,No,0
1,CUST_00000002,42.0,Male,Married,High School,Rural,14,Month-to-month,Credit card,Yes,...,0,9.0,Resolved,67.77,968.86,0.7366,0,2022-10-11,No,0
2,CUST_00000003,54.0,Female,Married,High School,Suburban,40,One year,Credit card,Yes,...,1,4.0,Resolved,99.68,3788.89,0.6515,1,2020-08-21,No,0
3,CUST_00000004,67.0,Female,Divorced,Master,Rural,3,Month-to-month,Electronic check,No,...,3,4.0,Resolved,87.57,239.07,0.6584,0,2023-09-26,Yes,0
4,CUST_00000005,41.0,Female,Married,Master,Suburban,11,Month-to-month,Credit card,Yes,...,1,6.0,Resolved,73.58,784.00,0.6454,1,2023-01-16,No,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,CUST_00014996,31.0,Female,Divorced,High School,Suburban,10,Month-to-month,Bank transfer,Yes,...,1,6.0,Escalated,57.51,604.07,0.6535,0,2023-02-09,No,0
14996,CUST_00014997,63.0,Female,Divorced,Bachelor,Suburban,31,Two year,Electronic check,Yes,...,0,7.0,Escalated,113.17,3378.87,2.1353,1,2021-05-09,No,0
14997,CUST_00014998,64.0,Female,Single,High School,Urban,5,Month-to-month,Electronic check,Yes,...,1,6.0,Escalated,49.56,249.88,0.5697,0,2023-07-11,Yes,0
14998,CUST_00014999,31.0,Male,Divorced,Master,Suburban,21,Month-to-month,Bank transfer,Yes,...,4,4.0,Escalated,125.88,2589.05,1.1042,1,2022-03-20,No,0
